# NLP Group 7 Project P2

## Load the dataset

In [ ]:
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("MathArena/final_answer_comps", split="train")
df = dataset.to_pandas()

## General info and null values

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   problem_idx   139 non-null    int64 
 1   answer        139 non-null    object
 2   problem_type  130 non-null    object
 3   problem       139 non-null    object
 4   competition   139 non-null    object
 5   source        9 non-null      object
dtypes: int64(1), object(5)
memory usage: 6.6+ KB


In [3]:
print("Dataset Head:\n", df.head(), sep="", end="\n" + "="*50 + "\n")
print("Dataset Columns:\n", df.columns, sep="", end="\n" + "="*50 + "\n")
print("Dataset Shape:\n", df.shape, sep="", end="\n" + "="*50 + "\n")

Dataset Head:
   problem_idx answer                    problem_type  \
0            1     70                 [Number Theory]   
1            2    588                      [Geometry]   
2            3     16                 [Combinatorics]   
3            4    117                       [Algebra]   
4            5    279  [Combinatorics, Number Theory]   

                                             problem     competition source  
0  Find the sum of all integer bases $b>9$ for wh...  aime/aime_2025   None  
1  On $\triangle ABC$ points $A, D, E$, and $B$ l...  aime/aime_2025   None  
2  The 9 members of a baseball team went to an ic...  aime/aime_2025   None  
3  Find the number of ordered pairs $(x,y)$, wher...  aime/aime_2025   None  
4  There are $8!= 40320$ eight-digit positive int...  aime/aime_2025   None  
Dataset Columns:
Index(['problem_idx', 'answer', 'problem_type', 'problem', 'competition',
       'source'],
      dtype='object')
Dataset Shape:
(139, 6)


## Communication with the model

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

ENDPOINT = "https://nlp-pcaf.services.ai.azure.com/openai/v1/"
MODEL_NAME = "DeepSeek-V3-0324"
DEPLOYMENT_NAME = "DeepSeek-V3-0324"

api_key = os.getenv("API_KEY")

client = OpenAI(
    base_url=f"{ENDPOINT}",
    api_key=api_key
)

completion = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?",
        }
    ],
)

print(completion.choices[0].message)

ChatCompletionMessage(content='The capital of France is **Paris**. It is one of the most famous and visited cities in the world, known for its iconic landmarks such as the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral. \n\nWould you like more details about Paris or France? 😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


In [5]:
def get_llm_response(role_prompt, user_input, temperature=0.0):
    """Sends a request to the Azure LLM with a System Role Prompt."""
    messages = [
        {"role": "system", "content": role_prompt},
        {"role": "user", "content": user_input}
    ]
    
    # Configure the response format for JSON
    response_format = {"type": "text"}
    if "JSON" in role_prompt or "JSON" in user_input:
         response_format = {"type": "json_object"}
    
    try:
        response = client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=messages,
            temperature=temperature,
            response_format=response_format
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"LLM API Error: {e}"